In [1]:
import sys
sys.path.append('/home/pavel/pyadi-iio/')
sys.path.append('/home/pavel/libiio')
import adi
import matplotlib.pyplot as plt
import numpy as np

import time

In [52]:
URI_TX = "ip:192.168.2.1"   # <-- IP передатчика
FS = int(1e6)                # sample rate
LO = int(2.19e9)             # carrier
N = 65536                    # length of cyclic buffer

# Три тона для канала TX0
f1, f2, f3 = 80e3, 120e3, 220e3
a1, a2, a3 = 1.0, 0, 0

# Три тона для канала TX1
g1, g2, g3 = -80e3, 160e3, 300e3
b1, b2, b3 = 1.0, 0.0, 0.0

In [ ]:
sdr = adi.ad9361(URI_TX)

sdr.sample_rate = FS
sdr.tx_lo = LO
sdr.tx_rf_bandwidth = int(FS)

# Включаем оба TX канала
sdr.tx_enabled_channels = [0, 1]


sdr.gain_control_mode_chan0 = "manual"
sdr.gain_control_mode_chan1 = "manual"
sdr.tx_hardwaregain_chan0 = int(20)
sdr.tx_hardwaregain_chan1 = int(20)
sdr.tx_cyclic_buffer = True

In [54]:
t = np.arange(N) / FS

# Канал TX0
tx0 = (
    a1 * np.exp(1j * 2 * np.pi * f1 * t) +
    a2 * np.exp(1j * 2 * np.pi * f2 * t) +
    a3 * np.exp(1j * 2 * np.pi * f3 * t)
)

# Канал TX1
tx1 = (
    b1 * np.exp(1j * 2 * np.pi * g1 * t) +
    b2 * np.exp(1j * 2 * np.pi * g2 * t) +
    b3 * np.exp(1j * 2 * np.pi * g3 * t)
)

# Нормировка отдельно по каналам
tx0 = 2**14 * tx0 / np.max(np.abs(tx0))
tx1 = 2**14 * tx1 / np.max(np.abs(tx1))

tx0 = tx0.astype(np.complex64)
tx1 = tx1.astype(np.complex64)

print(f"TX LO = {LO/1e9:.6f} GHz, FS = {FS/1e6:.3f} MSPS")
print("TX0 tones:", [f1, f2, f3])
print("TX1 tones:", [g1, g2, g3])


TX LO = 2.190000 GHz, FS = 1.000 MSPS
TX0 tones: [80000.0, 120000.0, 220000.0]
TX1 tones: [-80000.0, 160000.0, 300000.0]


In [56]:
# Для 2 каналов передаём СПИСОК массивов
sdr.tx([tx0, tx1])

Exception: TX buffer has been submitted in cyclic mode. To push more data the tx buffer must be destroyed first.

In [57]:
sdr.tx_destroy_buffer()